In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

In [2]:
data = pd.read_csv("../data/loan_approval.csv")
data.head()

,name,city,income,credit_score,loan_amount,years_employed,points,loan_approved
0,Allison Hill,East Jill,113810,389,39698,27,50.0,False
1,Brandon Hall,New Jamesside,44592,729,15446,28,55.0,False
2,Rhonda Smith,Lake Roberto,33278,584,11189,13,45.0,False
3,Gabrielle Davis,West Melanieview,127196,344,48823,29,50.0,False
4,Valerie Gray,Mariastad,66048,496,47174,4,25.0,False


In [3]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   name            2000 non-null   str    
 1   city            2000 non-null   str    
 2   income          2000 non-null   int64  
 3   credit_score    2000 non-null   int64  
 4   loan_amount     2000 non-null   int64  
 5   years_employed  2000 non-null   int64  
 6   points          2000 non-null   float64
 7   loan_approved   2000 non-null   bool   
dtypes: bool(1), float64(1), int64(4), str(2)
memory usage: 111.5 KB


In [4]:
def plot(data,cols):
    plot = px.histogram(data,y =cols,x = y,title=f"Loan Approved vs {cols}")
    plot.show()

for i in data:
    plot(data,i)

NameError: name 'y' is not defined

In [ ]:
x = data.drop(columns=["name","loan_approved"])
y = data["loan_approved"]

print(x)
print(y)

                  city  income  credit_score  loan_amount  years_employed  \
0            East Jill  113810           389        39698              27   
1        New Jamesside   44592           729        15446              28   
2         Lake Roberto   33278           584        11189              13   
3     West Melanieview  127196           344        48823              29   
4            Mariastad   66048           496        47174               4   
...                ...     ...           ...          ...             ...   
1995         Robertton   92163           770        12251              13   
1996         New Frank   38799           635        48259              17   
1997        East Haley   41957           763        16752               5   
1998          Adamland  139022           360        24031              35   
1999    New Nathantown   41188           482        31397               6   

      points  
0       50.0  
1       55.0  
2       45.0  
3       50.0  


In [ ]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)
print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

(1600, 6)
(400, 6)
(1600,)
(400,)


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.impute import SimpleImputer

num_features = x.select_dtypes(include=["int64","float64"]).columns
cat_features = x.select_dtypes(include=["object"]).columns

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_features),
    ("cat", cat_pipeline, cat_features)
])

C:\Users\nigam\AppData\Local\Temp\ipykernel_15216\3898222807.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_features = x.select_dtypes(include=["object"]).columns


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from sklearn.svm import SVC

models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "RandomForest": RandomForestClassifier(
        n_estimators=200,
        max_depth=50,
        n_jobs=-1,
        class_weight="balanced"
    ),
    "GradientBoosting": GradientBoostingClassifier(n_estimators=200),
    "SVM": SVC(class_weight="balanced")
}

In [ ]:
from sklearn.metrics import accuracy_score,f1_score,classification_report

results = []

try:
    for name, model in models.items():
        pipe = Pipeline([
            ("preprocessor", preprocessor),
            ("model", model)
        ])

        pipe.fit(x_train, y_train)
        pred = pipe.predict(x_test)

        f1 = f1_score(y_test, pred)
        accuracy_score(y_test,pred)

        print(f"\n{name}")
        print(classification_report(y_test, pred))

        results.append({
            "Model": name,
            "F1-Score": f1
        })
except Exception as e:
    print(f"{name} failed as {e}")
    
pd.DataFrame(results)            

,Model,Accuracy,F1-Score
0,Logistic,0.9975,0.99726
1,RandomForest,1.0000,1.00000
2,GradientBoosting,1.0000,1.00000
3,SVM,0.9925,0.99187


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred))

[[214   3]
 [  0 183]]
              precision    recall  f1-score   support

       False       1.00      0.99      0.99       217
        True       0.98      1.00      0.99       183

    accuracy                           0.99       400
   macro avg       0.99      0.99      0.99       400
weighted avg       0.99      0.99      0.99       400



In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(pipe, x, y, cv=5, scoring="f1")

print("F1 CV Score:", scores.mean())

F1 CV Score: 0.9926567872236483


## Best is random Forest Classifier 